# Client Load Exp for Shabdiz

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, iterable))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# # num_nodes = 4
# zone_no = 0

# # Use 2 extra 2-core machines as client machines.
# n_clients = 2

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = CLIENT_DURATION_SEC + 45
# CLIENT_TOTAL_REQUESTS = 100000000
# CLIENT_MAX_IN_FLIGHT = 400
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Throughput/latency load points.
# # This gives enough points for a throughput-vs-latency plot without too many subruns.
# LOAD_POINTS = [
#     # {"active_clients": 1, "client_threads": 1, "max_in_flight": 100},   # total inflight 100
#     {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},   # total inflight 150
#     # {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},   # total inflight 200
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
#     # {"active_clients": 1, "client_threads": 4, "max_in_flight": 100},   # total inflight 400
#     # {"active_clients": 1, "client_threads": 8, "max_in_flight": 100},   # total inflight 800
#     # {"active_clients": 2, "client_threads": 8, "max_in_flight": 100},   # total inflight 1600
# ]


# for num_nodes in [8]:
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     def get_zone_for_instance(i):
#         if i < int(num_nodes / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     # Fetch all tsm-sc-* instances across ALL zones
#     fetch_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     '''

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     print("\n➡ Existing instances to delete:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     def delete_instance(instance):
#         name, inst_zone = instance
#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     # if instances:
#     #     run_parallel(delete_instance, instances, max_workers=32)
#     #     print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
#     # else:
#     #     print("\n✔ No tsm-sc-* instances found.\n")

#     # Create commands list
#     commands = []

#     # Create replica nodes.
#     for i in range(num_nodes):
#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create tsm-sc-{i:03} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())

#     # Create client machines after replica nodes.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create tsm-sc-{client_idx:03} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     # run_parallel(run_command, commands, max_workers=48)

#     print("All instances launched.")

#     # Get sorted node and client IPs.
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#     n_collection = 100
#     subprocess.call('make -j8', shell=True)

#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     results = run_parallel(
#         kill_stellar_private,
#         range(num_nodes + n_clients),
#         max_workers=48
#     )

#     def git_pull_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     results = run_parallel(
#         git_pull_stellar,
#         range(num_nodes + n_clients),
#         max_workers=48
#     )
#     print(results)

#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     target_file = "../stellar-private/node2/stellar-core.cfg"
#     line_to_add = "MEMORY_PROF=true"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     def compile_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     # results = run_parallel(
#     #     compile_stellar,
#     #     range(num_nodes + n_clients),
#     #     max_workers=48
#     # )
#     print(results)

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # No sleep intervals. SEND_INTERVAL_US is fixed at 0.
#     # We vary client concurrency to create the throughput-vs-latency points.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]
#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight
        
#         total_client_threads = active_clients * client_threads

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=48
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(60)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"shab_tput_latency_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={CLIENT_MAX_IN_FLIGHT}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, num_nodes)),
#             max_workers=48
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# Main num nodes experiment

In [7]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
from pathlib import Path


def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


# latencies: 50, 90 150, 210
default_region = ['us-central1-a']
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# Regions

zone_no = 0

# Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [48, 32, 16, 8, 4]
# NUM_NODES_LIST = [32]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Start clean only once, before the largest run.
# After that, keep the lower-index VMs and delete only the extra higher-index VMs.
DELETE_BEFORE_FIRST_RUN = True

# Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# that will not be needed by the next smaller run.
DELETE_UNUSED_AFTER_EACH_RUN = True

# Delete the remaining 4 replica VMs + 1 client VM after the final run.
DELETE_ALL_AFTER_FINAL_RUN = True

# Code is unchanged across node-count runs, so compile only on the first run.
COMPILE_ONLY_FIRST_RUN = True

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
]


for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# for zone_no in  [0,1,2,3, 4]:

    project = "research-488322"
    zone = "us-central1-a"
    machine_type = "e2-standard-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"

    print("\n" + "#" * 100)
    print(f"Starting experiment for num_nodes={num_nodes}")
    print("#" * 100)

    def get_zone_for_instance(i):
        # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
        # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
        # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
        if i < int(MAX_NUM_NODES / 2):
            return default_region[0]
        else:
            return regions[zone_no]

    def fetch_existing_instances():
        fetch_cmd = f'''
        gcloud compute instances list \
            --project={project} \
            --filter="name~'^tsm-sc-'" \
            --format="value(name,zone)"
        '''

        output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
        instances = []

        for line in output.splitlines():
            if line.strip():
                name, inst_zone = line.split()
                instances.append((name, inst_zone))

        return instances

    def delete_instance(instance):
        name, inst_zone = instance

        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={inst_zone} \
            --project={project} \
            --quiet
        '''

        print(f"🗑️ Deleting {name} in {inst_zone}")
        return subprocess.call(cmd, shell=True)

    def parse_tsm_index(name):
        return int(name.rsplit("-", 1)[1])

    # -------------------------------------------------------------------------
    # Delete existing tsm-sc-* instances only before the first/largest run.
    # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
    # -------------------------------------------------------------------------
    if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
        instances = fetch_existing_instances()

        print("\n➡ Existing instances to delete before first run:")
        for name, inst_zone in instances:
            print(f"  - {name} ({inst_zone})")

        if instances:
            run_parallel(
                delete_instance,
                instances,
                max_workers=min(32, len(instances))
            )
            print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
        else:
            print("\n✔ No existing tsm-sc-* instances found before first run.\n")

    # -------------------------------------------------------------------------
    # Create only the missing replica/client machines for this num_nodes run.
    # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
    # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
    # Client:   tsm-sc-num_nodes
    # -------------------------------------------------------------------------
    existing_instances = fetch_existing_instances()
    existing_names = {name for name, _ in existing_instances}

    commands = []
    newly_created_indices = []

    # Create replica nodes only if they do not already exist.
    for i in range(num_nodes):
        instance_name = f"tsm-sc-{i:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing replica {instance_name}")
            continue

        inst_zone = get_zone_for_instance(i)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(i)

    # Create client machine only if it does not already exist.
    for i in range(n_clients):
        client_idx = num_nodes + i
        instance_name = f"tsm-sc-{client_idx:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing client {instance_name}")
            continue

        inst_zone = get_zone_for_instance(client_idx)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(client_idx)

    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)

    if commands:
        run_parallel(
            run_command,
            commands,
            max_workers=min(48, len(commands))
        )

        print("All missing instances launched.")

        # Give GCP/SSH a little time after VM creation.
        time.sleep(30)
    else:
        print("✔ All required instances already exist; no VM creation needed.")

    # -------------------------------------------------------------------------
    # Get sorted node and client IPs.
    # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
    # Client IPs must not be included.
    # -------------------------------------------------------------------------
    ip_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --sort-by=name \
        --format="value(name,zone,networkInterfaces[0].networkIP)"
    '''

    ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

    instance_records = []

    for line in ip_output.splitlines():
        if line.strip():
            name, inst_zone, ip = line.split()
            idx = int(name.rsplit("-", 1)[1])
            instance_records.append((idx, name, inst_zone, ip))

    instance_records.sort()

    node_records = [r for r in instance_records if r[0] < num_nodes]
    client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

    if len(node_records) != num_nodes:
        raise RuntimeError(
            f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
        )

    if len(client_records) != n_clients:
        raise RuntimeError(
            f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
        )

    iplist = [r[3] for r in node_records]

    with open("tsm_ips.txt", "w") as f:
        for ip in iplist:
            f.write(ip + "\n")

    print("🎯 Node IPs:", iplist)
    print("🎯 Client instances:", client_records)

    node1_ip = iplist[0]
    print(f"Clients will connect to leader/node1 at: {node1_ip}")

    # -------------------------------------------------------------------------
    # Push/pull/compile only for the first run.
    # Later runs reuse the same lower-index VMs, and the code has not changed.
    # -------------------------------------------------------------------------
    def kill_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = f"""\
cd /home/tejas/stellar-private; \
sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

    if do_compile_this_run:
        subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

        n_collection = 100
        subprocess.call('make -j8', shell=True)

        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        def git_pull_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )
        print(results)

        # ---------------------------------------------------------------------
        # Compile only on the first/largest run.
        # Since later runs reuse a subset of these VMs, no recompilation is needed.
        # ---------------------------------------------------------------------
        def compile_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )
        print(results)
    else:
        print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

    # -------------------------------------------------------------------------
    # Generate stellar-private configs locally using only replica IPs.
    # -------------------------------------------------------------------------
    stellar_private_path = Path('../stellar-private')
    if stellar_private_path.exists():
        shutil.rmtree(stellar_private_path)
    stellar_private_path.mkdir()

    subprocess.call(
        'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
        shell=True
    )

    subprocess.call(
        'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
        './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
        shell=True
    )

    # Enable custom message only on leader.
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg"

    subprocess.call(
        f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
        shell=True
    )

    # Optional memory profiling on node2.
    # if num_nodes >= 2:
    #     target_file = "../stellar-private/node2/stellar-core.cfg"
    #     line_to_add = "MEMORY_PROF=true"

    #     subprocess.call(
    #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
    #         shell=True
    #     )

    #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

    # -------------------------------------------------------------------------
    # Throughput/latency experiment loop.
    # Fixed offered load for scalability:
    # active_clients=1, client_threads=2, max_in_flight=100.
    # Aggregate max in-flight = 200.
    # -------------------------------------------------------------------------
    for load in LOAD_POINTS:
        active_clients = load["active_clients"]
        client_threads = load["client_threads"]
        client_max_in_flight = load["max_in_flight"]

        if active_clients > n_clients:
            raise RuntimeError(
                f"active_clients={active_clients} exceeds n_clients={n_clients}"
            )

        total_client_threads = active_clients * client_threads
        aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

        run_label = (
            f"clients_{active_clients}_threads_{client_threads}_"
            f"inflight_{client_max_in_flight}_"
            f"total_threads_{total_client_threads}_"
            f"total_inflight_{aggregate_max_in_flight}"
        )

        print("\n" + "=" * 80)
        print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
        print("=" * 80)

        def clean_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = f"""\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        results = run_parallel(
            clean_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        def copy_folder_to_instance(
            i,
            source_folder="/home/tejas/stellar-private",
            destination_path="/home/tejas/stellar-private"
        ):
            inst_zone = get_zone_for_instance(i)
            instance_name = f"tsm-sc-{i:03}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

            print(f"Executing command for {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Command for {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        results = run_parallel(
            copy_folder_to_instance,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        def run_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            node_number = i + 1
            instance_name = f"tsm-sc-{i:03}"

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # Kill old processes on all replica and client machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        # Start consensus replicas.
        results = run_parallel(
            run_stellar_private,
            range(num_nodes),
            max_workers=min(48, num_nodes)
        )

        print(results)
        print("All Stellar nodes should be starting in the background.")

        # Give nodes time to authenticate and start the client listener.
        time.sleep(80)

        def run_stellar_client(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing client command: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # Start only the required number of client VMs for this load point.
        active_client_indices = [
            num_nodes + j for j in range(active_clients)
        ]

        results = run_parallel(
            run_stellar_client,
            active_client_indices,
            max_workers=active_clients
        )

        print(results)
        print(
            f"Started {active_clients} client VM(s), "
            f"each with {client_threads} client threads. "
            f"Total client threads = {total_client_threads}. "
            f"Aggregate max in-flight = {aggregate_max_in_flight}."
        )

        # Wait for the duration run to produce stable per-second client logs.
        # The clients may wait forever on final partial batches, so we kill them after this.
        time.sleep(CLIENT_WAIT_AFTER_START_SEC)

        # Stop all nodes and clients.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        remote_base_folder = "/home/tejas/stellar-private"

        local_base_destination = (
            "/home/tejas/work/experiments/shabdiz/"
            + f"PBFT_vs_num_nodes_{num_nodes}_{run_label}"
        )

        Path(local_base_destination).mkdir(parents=True, exist_ok=True)

        # Save run metadata.
        with open(Path(local_base_destination) / "run_config.txt", "w") as f:
            f.write(f"num_nodes={num_nodes}\n")
            f.write(f"active_clients={active_clients}\n")
            f.write(f"client_threads_per_vm={client_threads}\n")
            f.write(f"total_client_threads={total_client_threads}\n")
            f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
            f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
            f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
            f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
            f.write(f"leader_ip={node1_ip}\n")
            f.write(f"machine_type={machine_type}\n")
            f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

        def copy_folder_from_instance(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"

            node_number = i + 1
            node_folder = f"node{node_number}"

            remote_source_path = posixpath.join(remote_base_folder, node_folder)

            local_destination_path = Path(local_base_destination) / instance_name
            local_destination_path.mkdir(parents=True, exist_ok=True)

            remote_source = f"{instance_name}:{remote_source_path}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

            print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Copy from {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        # Copy only a few node logs to reduce time.
        # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
        node_copy_results = run_parallel(
            copy_folder_from_instance,
            range(min(3, num_nodes)),
            max_workers=min(48, max(1, min(3, num_nodes)))
        )

        def copy_client_log(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_source = (
                f"{instance_name}:/home/tejas/stellar-private/"
                f"stellar-client-{client_id}.log"
            )

            local_destination_path = Path(local_base_destination)
            local_destination_path.mkdir(parents=True, exist_ok=True)

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

            print(f"Copying client log from {instance_name}...")

            output = subprocess.call(command, shell=True)

            print(f"Copy finished with exit code: {output}")

            return (instance_name, output)

        client_copy_results = run_parallel(
            copy_client_log,
            active_client_indices,
            max_workers=active_clients
        )

        print("\n--- Summary of Download Results ---")
        print("Node log copies:", node_copy_results)
        print("Client log copies:", client_copy_results)
        print(f"Saved run to: {local_base_destination}")

    # -------------------------------------------------------------------------
    # After this run, delete only the machines that the next smaller run will
    # not need. Example:
    #   after 48-node run, keep 000..032 and delete 033..048
    #   after 32-node run, keep 000..016 and delete 017..032
    #   after 16-node run, keep 000..008 and delete 009..016
    #   after 8-node run,  keep 000..004 and delete 005..008
    # Final run optionally deletes everything.
    # -------------------------------------------------------------------------
    if DELETE_UNUSED_AFTER_EACH_RUN:
        instances = fetch_existing_instances()

        if run_idx + 1 < len(NUM_NODES_LIST):
            next_num_nodes = NUM_NODES_LIST[run_idx + 1]
            keep_count = next_num_nodes + n_clients

            instances_to_delete = [
                (name, inst_zone)
                for name, inst_zone in instances
                if parse_tsm_index(name) >= keep_count
            ]

            print(
                f"\n➡ After num_nodes={num_nodes}, next run needs "
                f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
            )
        elif DELETE_ALL_AFTER_FINAL_RUN:
            instances_to_delete = instances

            print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
        else:
            instances_to_delete = []

            print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

        for name, inst_zone in instances_to_delete:
            print(f"  - {name} ({inst_zone})")

        if instances_to_delete:
            run_parallel(
                delete_instance,
                instances_to_delete,
                max_workers=min(48, len(instances_to_delete))
            )
            print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
        else:
            print("\n✔ No unneeded instances to delete after this run.\n")



####################################################################################################
Starting experiment for num_nodes=32
####################################################################################################



➡ Existing instances to delete before first run:

✔ No existing tsm-sc-* instances found before first run.



Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-a             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm             

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-011].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-011  us-central1-a  e2-standard-2               10.128.0.80  35.238.22.10  RUNNING


...

✅ 32-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node5/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node6/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node7/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node8/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf 

ssh: connect to host 34.45.254.140 port 22: Connection refused

Recommendation: To check for possible causes of SSH connectivity issues and get
recommendations, rerun the ssh command with the --troubleshoot option.

gcloud compute ssh tsm-sc-025 --project=research-488322 --zone=us-central1-a --troubleshoot

Or, to investigate an IAP tunneling issue:

gcloud compute ssh tsm-sc-025 --project=research-488322 --zone=us-central1-a --troubleshoot --tunnel-through-iap

ERROR: (gcloud.compute.ssh) [/usr/bin/ssh] exited with return code [255].


Return code for tsm-sc-025: 255
Return code for tsm-sc-004: 1
Return code for tsm-sc-018: 1
Return code for tsm-sc-011: 1
Return code for tsm-sc-032: 0
Return code for tsm-sc-007: 1
Return code for tsm-sc-029: 1
Return code for tsm-sc-008: 1
Return code for tsm-sc-006: 1
Return code for tsm-sc-019: 1
Return code for tsm-sc-001: 1
Return code for tsm-sc-000: 1
Return code for tsm-sc-017: 1
Return code for tsm-sc-014: 1
Return code for tsm-sc-013: 1
Return code for tsm-sc-021: 1
Return code for tsm-sc-002: 1
Return code for tsm-sc-016: 1
Return code for tsm-sc-003: 1
Return code for tsm-sc-024: 1
Return code for tsm-sc-027: 1
Return code for tsm-sc-023: 1
Return code for tsm-sc-028: 1
Return code for tsm-sc-015: 1
Return code for tsm-sc-022: 1
Return code for tsm-sc-005: 1
Return code for tsm-sc-020: 1
Return code for tsm-sc-010: 1
Return code for tsm-sc-012: 1
Return code for tsm-sc-031: 1
Return code for tsm-sc-030: 1
Return code for tsm-sc-009: 1
Return code for tsm-sc-026: 1
Executin

Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-027].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-010].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-005].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-009].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-008].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-007].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-031].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us


🧹 Deleted 33 unneeded instance(s) after num_nodes=32.



# Failure Experiment

In [28]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [16]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 180
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         # results = run_parallel(
#         #     compile_stellar,
#         #     range(num_nodes + n_clients),
#         #     max_workers=min(48, num_nodes + n_clients)
#         # )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     # if num_nodes >= 2:
#     #     target_file = "../stellar-private/node2/stellar-core.cfg"
#     #     line_to_add = "MEMORY_PROF=true"

#     #     subprocess.call(
#     #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#     #         shell=True
#     #     )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         # time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         time.sleep(100)

#         results = run_parallel(
#             kill_stellar_private,
#             [4],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)

        


#         results = run_parallel(
#             kill_stellar_private,
#             [5],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)


#         results = run_parallel(
#             kill_stellar_private,
#             [6],
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [7],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [8],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(80)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"Failure_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # After this run, delete only the machines that the next smaller run will
#     # not need. Example:
#     #   after 48-node run, keep 000..032 and delete 033..048
#     #   after 32-node run, keep 000..016 and delete 017..032
#     #   after 16-node run, keep 000..008 and delete 009..016
#     #   after 8-node run,  keep 000..004 and delete 005..008
#     # Final run optionally deletes everything.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(32, len(instances_to_delete))
#             )
#             print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")



####################################################################################################
Starting experiment for num_nodes=16
####################################################################################################
✔ Reusing existing replica tsm-sc-000
✔ Reusing existing replica tsm-sc-001
✔ Reusing existing replica tsm-sc-002
✔ Reusing existing replica tsm-sc-003
✔ Reusing existing replica tsm-sc-004
✔ Reusing existing replica tsm-sc-005
✔ Reusing existing replica tsm-sc-006
✔ Reusing existing replica tsm-sc-007
✔ Reusing existing replica tsm-sc-008
✔ Reusing existing replica tsm-sc-009
✔ Reusing existing replica tsm-sc-010
✔ Reusing existing replica tsm-sc-011
✔ Reusing existing replica tsm-sc-012
✔ Reusing existing replica tsm-sc-013
✔ Reusing existing replica tsm-sc-014
✔ Reusing existing replica tsm-sc-015
✔ Reusing existing client tsm-sc-016
✔ All required instances already exist; no VM creation needed.
🎯 Node IPs: ['10.128.0.108', '10.128.0.109', '10.128

To github.com:tejas-shivanand-mane/stellar-core.git
   a58b4b8..61d602b  main -> main


make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/contrib'
Making all in dist-build
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
Making all in msvc-scripts
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/msvc-

From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main


Updating a58b4b8..61d602b
Fast-forward
Updating a58b4b8..61d602b
Fast-forward
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
Updating a58b4b8..61d602b
Fast-forward
Updating a58b4b8..61d602b
Fast-forward
Updating a58b4b8..61d602b
Fast-forward
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 361

...

Initializing database for node2...
Initializing database for node3...
Initializing database for node4...
Initializing database for node5...
Initializing database for node6...
Initializing database for node7...
Initializing database for node8...


2026-06-23T01:01:07.426 [default INFO] Config from /home/tejas/stellar-private/node6/stellar-core.cfg
2026-06-23T01:01:07.429 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "node14",
      "node1",
      "GAPE6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
      "node7",
      "node13",
      "node10",
      "node16",
      "node11",
      "node15",
      "node4"
   ]
}

2026-06-23T01:01:07.429 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-23T01:01:07.429 [default INFO] Assigning calculated value of 7 to FAILURE_SAFETY
2026-06-23T01:01:07.461 [default INFO] Config from /home/tejas/stellar-private/node7/stellar-core.cfg
2026-06-23T01:01:07.463 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "node14",
      "node1",
      "node6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
     

Initializing database for node9...
Initializing database for node10...
Initializing database for node11...
Initializing database for node12...
Initializing database for node13...
Initializing database for node14...
Initializing database for node15...


2026-06-23T01:01:07.657 [default INFO] Config from /home/tejas/stellar-private/node13/stellar-core.cfg
2026-06-23T01:01:07.660 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "node14",
      "node1",
      "node6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
      "node7",
      "GBRMK",
      "node10",
      "node16",
      "node11",
      "node15",
      "node4"
   ]
}

2026-06-23T01:01:07.660 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-23T01:01:07.660 [default INFO] Assigning calculated value of 7 to FAILURE_SAFETY
2026-06-23T01:01:07.689 [default INFO] Config from /home/tejas/stellar-private/node14/stellar-core.cfg
2026-06-23T01:01:07.692 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "GAJTT",
      "node1",
      "node6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
     

Initializing database for node16...
✅ 16-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node5/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node6/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node7/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node8/stellar-core.cfg &
/home/tejas/stell

# Collection vs Collection Rounds Experiment

In [26]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# import re
# from pathlib import Path


# # ============================================================
# # Helpers
# # ============================================================

# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []

#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# def run_command(command):
#     print(f"Running: {command}")
#     return subprocess.call(command, shell=True)


# # ============================================================
# # Experiment config
# # ============================================================

# # latencies: 50, 90, 150, 210
# default_region = ["us-central1-a"]
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
# regions = ["us-central1-a", "us-central1-a", "us-central1-a", "us-central1-a"]

# zone_no = 0

# # System size for this failure / collection-window experiment.
# NUM_NODES_LIST = [16]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Collection-round sweep.
# COLLECT_ATTEMPT_POINTS = [50]

# # Source file to patch before each subrun.
# OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# # Start clean only once, before the first run.
# DELETE_BEFORE_FIRST_RUN = False

# # Delete unused VMs after each run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete remaining VMs after final run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # IMPORTANT:
# # Code changes across collection-round settings, so we must compile every subrun.
# COMPILE_ONLY_FIRST_RUN = False

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 250
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed offered load.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
# ]



# # ============================================================
# # Local source patching
# # ============================================================

# def patch_max_collect_attempts(max_collect_attempts):
#     text = OVERLAY_MANAGER_IMPL_PATH.read_text()

#     pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
#     replacement = (
#         f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
#     )

#     new_text, count = re.subn(
#         pattern,
#         replacement,
#         text,
#         flags=re.MULTILINE,
#     )

#     if count != 1:
#         raise RuntimeError(
#             f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
#             f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
#         )

#     OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

#     print(
#         f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
#         f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
#     )


# def push_collect_patch_to_git(max_collect_attempts):
#     cmd = (
#         f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
#         f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
#         f'git push'
#     )

#     return subprocess.call(cmd, shell=True)


# # ============================================================
# # Main experiment loop
# # ============================================================

# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # Use MAX_NUM_NODES so zone assignment stays stable when VMs are reused.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first run, if enabled.
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances)),
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only missing replica/client machines for this num_nodes run.
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands)),
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # tsm_ips.txt must include only replica IPs, not clients.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [
#         r for r in instance_records
#         if num_nodes <= r[0] < num_nodes + n_clients
#     ]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Remote helper functions.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = """\
# sudo pkill -9 stellar-core || true; \
# sudo pkill -9 shab_client || true; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     def git_pull_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     def compile_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     def clean_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = """\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     def copy_folder_to_instance(
#         i,
#         source_folder="/home/tejas/stellar-private",
#         destination_path="/home/tejas/stellar-private",
#     ):
#         inst_zone = get_zone_for_instance(i)
#         instance_name = f"tsm-sc-{i:03}"

#         command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#         print(f"Executing command for {instance_name}: {command}")

#         output = subprocess.call(command, shell=True)

#         print(f"Command for {instance_name} finished with exit code: {output}")

#         return (instance_name, output)

#     def run_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         node_number = i + 1
#         instance_name = f"tsm-sc-{i:03}"

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")

#         output = subprocess.call(command, shell=True)

#         print(f"Return code for {instance_name}: {output}")
#         return output

#     def run_stellar_client(i, client_max_in_flight, client_threads):
#         inst_zone = get_zone_for_instance(i)

#         instance_name = f"tsm-sc-{i:03}"
#         client_id = i - num_nodes

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing client command: {command}")

#         output = subprocess.call(command, shell=True)

#         print(f"Return code for {instance_name}: {output}")
#         return output

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # This does not depend on MAX_COLLECT_ATTEMPTS, so we do it once per num_nodes.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path("../stellar-private")
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     subprocess.call(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True,
#     )

#     # -------------------------------------------------------------------------
#     # Collection-round sweep.
#     # -------------------------------------------------------------------------
#     for max_collect_attempts in COLLECT_ATTEMPT_POINTS:
#         print("\n" + "*" * 100)
#         print(f"Preparing subrun with MAX_COLLECT_ATTEMPTS={max_collect_attempts}")
#         print("*" * 100)

#         # Patch local source.
#         patch_max_collect_attempts(max_collect_attempts)

#         # Push patched source.
#         push_collect_patch_to_git(max_collect_attempts)

#         # Stop any old processes.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients),
#         )
#         print(results)

#         # Pull patched code on active VMs.
#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients),
#         )
#         print(results)

#         # Compile patched code on active VMs.
#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients),
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Throughput/latency loop.
#         # ---------------------------------------------------------------------
#         for load in LOAD_POINTS:
#             active_clients = load["active_clients"]
#             client_threads = load["client_threads"]
#             client_max_in_flight = load["max_in_flight"]

#             if active_clients > n_clients:
#                 raise RuntimeError(
#                     f"active_clients={active_clients} exceeds n_clients={n_clients}"
#                 )

#             total_client_threads = active_clients * client_threads
#             aggregate_max_in_flight = (
#                 active_clients * client_threads * client_max_in_flight
#             )

#             run_label = (
#                 f"clients_{active_clients}_threads_{client_threads}_"
#                 f"inflight_{client_max_in_flight}_"
#                 f"total_threads_{total_client_threads}_"
#                 f"total_inflight_{aggregate_max_in_flight}"
#             )

#             print("\n" + "=" * 80)
#             print(
#                 f"🚀 Starting run: num_nodes={num_nodes}, "
#                 f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, {run_label}"
#             )
#             print("=" * 80)

#             # Clean remote stellar-private.
#             results = run_parallel(
#                 clean_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Copy fresh stellar-private config to all active machines.
#             results = run_parallel(
#                 copy_folder_to_instance,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Kill old processes on all active machines.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Start consensus replicas.
#             results = run_parallel(
#                 run_stellar_private,
#                 range(num_nodes),
#                 max_workers=min(48, num_nodes),
#             )

#             print(results)
#             print("All Stellar nodes should be starting in the background.")

#             # Give nodes time to authenticate and start the client listener.
#             time.sleep(60)

#             # Start only required client VMs for this load point.
#             active_client_indices = [
#                 num_nodes + j for j in range(active_clients)
#             ]

#             results = run_parallel(
#                 lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print(results)
#             print(
#                 f"Started {active_clients} client VM(s), "
#                 f"each with {client_threads} client threads. "
#                 f"Total client threads = {total_client_threads}. "
#                 f"Aggregate max in-flight = {aggregate_max_in_flight}."
#             )

#             # Let the system run before forcing failures.
#             time.sleep(CLIENT_WAIT_AFTER_START_SEC)


#             # Stop all nodes and clients.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # -----------------------------------------------------------------
#             # Save logs.
#             # -----------------------------------------------------------------
#             remote_base_folder = "/home/tejas/stellar-private"

#             local_base_destination = (
#                 "/home/tejas/work/experiments/shabdiz/"
#                 + f"Collection_{max_collect_attempts}_"
#                 + f"nodes_{num_nodes}_{run_label}"
#             )

#             Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#             # Save run metadata.
#             with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#                 f.write(f"num_nodes={num_nodes}\n")
#                 f.write(f"max_collect_attempts={max_collect_attempts}\n")
#                 f.write(f"force_collect_after_sec=100\n")
#                 f.write(f"active_clients={active_clients}\n")
#                 f.write(f"client_threads_per_vm={client_threads}\n")
#                 f.write(f"total_client_threads={total_client_threads}\n")
#                 f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#                 f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#                 f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#                 f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#                 f.write(f"leader_ip={node1_ip}\n")
#                 f.write(f"machine_type={machine_type}\n")
#                 f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#             def copy_folder_from_instance(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"

#                 node_number = i + 1
#                 node_folder = f"node{node_number}"

#                 remote_source_path = posixpath.join(remote_base_folder, node_folder)

#                 local_destination_path = Path(local_base_destination) / instance_name
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 remote_source = f"{instance_name}:{remote_source_path}"

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#                 print(
#                     f"Executing command to copy {node_folder} "
#                     f"from {instance_name}: {command}"
#                 )

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy from {instance_name} finished with exit code: {output}")

#                 return (instance_name, output)

#             # Copy all replica logs.
#             node_copy_results = run_parallel(
#                 copy_folder_from_instance,
#                 range(3),
#                 max_workers=min(48, max(1, num_nodes)),
#             )

#             def copy_client_log(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"
#                 client_id = i - num_nodes

#                 remote_source = (
#                     f"{instance_name}:/home/tejas/stellar-private/"
#                     f"stellar-client-{client_id}.log"
#                 )

#                 local_destination_path = Path(local_base_destination)
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#                 print(f"Copying client log from {instance_name}...")

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy finished with exit code: {output}")

#                 return (instance_name, output)

#             client_copy_results = run_parallel(
#                 copy_client_log,
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print("\n--- Summary of Download Results ---")
#             print("Node log copies:", node_copy_results)
#             print("Client log copies:", client_copy_results)
#             print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # Optional VM cleanup after this num_nodes run.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(32, len(instances_to_delete)),
#             )
#             print(
#                 f"\n🧹 Deleted {len(instances_to_delete)} "
#                 f"unneeded instance(s) after num_nodes={num_nodes}.\n"
#             )
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")


####################################################################################################
Starting experiment for num_nodes=16
####################################################################################################


Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-a             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm             

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-015].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-015  us-central1-a  e2-standard-2               10.128.0.25  34.10.149.202  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-008].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-010].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



...

/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket/HotArchiveBucketList.o bucket/InMemoryIndex.o bucket/LiveBucket.o bucket/LiveBucketIndex.o bucket/LiveBucketList.o bucket/MergeKey.o bucket/SearchableBucketList.o catchup/ApplyBucketsWork.o catchup/ApplyBufferedLedgersWork.o catchup/ApplyCheckpointWork.o catchup/ApplyLedgerWork.o

At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics


/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket/HotArchiveBucketList.o bucket/InMemoryIndex.o bucket/LiveBucket.o bucket/LiveBucketIndex.o bucket/LiveBucketList.o bucket/MergeKey.o bucket/SearchableBucketList.o catchup/ApplyBucketsWork.o catchup/ApplyBufferedLedgersWork.o catchup/ApplyCheckpointWork.o catchup/ApplyLedgerWork.o

At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics
At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics


/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket/HotArchiveBucketList.o bucket/InMemoryIndex.o bucket/LiveBucket.o bucket/LiveBucketIndex.o bucket/LiveBucketList.o bucket/MergeKey.o bucket/SearchableBucketList.o catchup/ApplyBucketsWork.o catchup/ApplyBufferedLedgersWork.o catchup/ApplyCheckpointWork.o catchup/ApplyLedgerWork.o

In [23]:
#             # Clean remote stellar-private.
#             results = run_parallel(
#                 clean_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Copy fresh stellar-private config to all active machines.
#             results = run_parallel(
#                 copy_folder_to_instance,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Kill old processes on all active machines.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Start consensus replicas.
#             results = run_parallel(
#                 run_stellar_private,
#                 range(num_nodes),
#                 max_workers=min(48, num_nodes),
#             )

#             print(results)
#             print("All Stellar nodes should be starting in the background.")

#             # Give nodes time to authenticate and start the client listener.
#             time.sleep(60)

#             # Start only required client VMs for this load point.
#             active_client_indices = [
#                 num_nodes + j for j in range(active_clients)
#             ]

#             results = run_parallel(
#                 lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print(results)
#             print(
#                 f"Started {active_clients} client VM(s), "
#                 f"each with {client_threads} client threads. "
#                 f"Total client threads = {total_client_threads}. "
#                 f"Aggregate max in-flight = {aggregate_max_in_flight}."
#             )

#             # Let the system run before forcing failures.
#             time.sleep(CLIENT_WAIT_AFTER_START_SEC)


#             # Stop all nodes and clients.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # -----------------------------------------------------------------
#             # Save logs.
#             # -----------------------------------------------------------------
#             remote_base_folder = "/home/tejas/stellar-private"

#             local_base_destination = (
#                 "/home/tejas/work/experiments/shabdiz/"
#                 + f"Collection_{max_collect_attempts}_"
#                 + f"nodes_{num_nodes}_{run_label}"
#             )

#             Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#             # Save run metadata.
#             with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#                 f.write(f"num_nodes={num_nodes}\n")
#                 f.write(f"max_collect_attempts={max_collect_attempts}\n")
#                 f.write(f"force_collect_after_sec=100\n")
#                 f.write(f"active_clients={active_clients}\n")
#                 f.write(f"client_threads_per_vm={client_threads}\n")
#                 f.write(f"total_client_threads={total_client_threads}\n")
#                 f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#                 f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#                 f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#                 f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#                 f.write(f"leader_ip={node1_ip}\n")
#                 f.write(f"machine_type={machine_type}\n")
#                 f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#             def copy_folder_from_instance(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"

#                 node_number = i + 1
#                 node_folder = f"node{node_number}"

#                 remote_source_path = posixpath.join(remote_base_folder, node_folder)

#                 local_destination_path = Path(local_base_destination) / instance_name
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 remote_source = f"{instance_name}:{remote_source_path}"

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#                 print(
#                     f"Executing command to copy {node_folder} "
#                     f"from {instance_name}: {command}"
#                 )

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy from {instance_name} finished with exit code: {output}")

#                 return (instance_name, output)

#             # Copy all replica logs.
#             node_copy_results = run_parallel(
#                 copy_folder_from_instance,
#                 range(3),
#                 max_workers=min(48, max(1, num_nodes)),
#             )

#             def copy_client_log(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"
#                 client_id = i - num_nodes

#                 remote_source = (
#                     f"{instance_name}:/home/tejas/stellar-private/"
#                     f"stellar-client-{client_id}.log"
#                 )

#                 local_destination_path = Path(local_base_destination)
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#                 print(f"Copying client log from {instance_name}...")

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy finished with exit code: {output}")

#                 return (instance_name, output)

#             client_copy_results = run_parallel(
#                 copy_client_log,
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print("\n--- Summary of Download Results ---")
#             print("Node log copies:", node_copy_results)
#             print("Client log copies:", client_copy_results)
#             print(f"Saved run to: {local_base_destination}")

Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-000" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-001" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-002" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-003" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-004" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-005" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-006" --project "research-48

In [28]:

# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# import re
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Fixed read-ratio experiment configuration
# # =============================================================================

# # Fixed system size for performance-vs-read-ratio experiment.
# NUM_NODES = 8

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = 8, client is tsm-sc-008.
# n_clients = 1

# # Read-ratio workloads. These must already exist in CustomYCSBWorkload.h:
# # WORKLOAD_R0, WORKLOAD_R25, WORKLOAD_R50, WORKLOAD_R75, WORKLOAD_R100
# READ_RATIO_WORKLOADS = [
#     # {"read_ratio": 0,   "workload": "WORKLOAD_R0"},
#     {"read_ratio": 10,   "workload": "WORKLOAD_R10"},
#     # {"read_ratio": 25,  "workload": "WORKLOAD_R25"},
#     # {"read_ratio": 50,  "workload": "WORKLOAD_R50"},
#     # {"read_ratio": 75,  "workload": "WORKLOAD_R75"},
#     # {"read_ratio": 20,  "workload": "WORKLOAD_R20"},
#     {"read_ratio": 40,  "workload": "WORKLOAD_R40"},
#     # {"read_ratio": 60,  "workload": "WORKLOAD_R60"},
#     # {"read_ratio": 80,  "workload": "WORKLOAD_R80"},
#     # {"read_ratio": 100, "workload": "WORKLOAD_R100"},
# ]

# # The file containing:
# #   static YCSBWorkload currentWorkload = WORKLOAD_R50;
# YCSB_HEADER_LOCAL = Path("src/overlay/CustomYCSBWorkload.h")

# # Start clean before the first read-ratio run.
# DELETE_BEFORE_FIRST_RUN = True

# # Delete all VMs after the final read-ratio run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile stellar-core only in the first read-ratio subrun.
# # The client is compiled every subrun because CustomYCSBWorkload.h changes.
# COMPILE_STELLAR_CORE_ONLY_FIRST_RUN = True

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 210
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Same fixed client load as your previous experiment.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]

# # GCP settings.
# project = "research-488322"
# zone = "us-central1-c"
# machine_type = "e2-standard-2"
# image_family = "tsm-sc-family"  # your custom image
# subnet = "default"
# gcp_username = "tejas"

# # latencies: 50, 90 150, 210
# default_region = ["us-central1-a"]
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
# regions = ["us-central1-a", "us-central1-a", "us-central1-a", "us-central1-a"]

# zone_no = 0


# def get_zone_for_instance(i):
#     # Keep the same placement rule as your previous script.
#     if i < int(NUM_NODES / 2):
#         return default_region[0]
#     else:
#         return regions[zone_no]


# def set_local_ycsb_workload(workload_name):
#     """
#     Updates this line in src/overlay/CustomYCSBWorkload.h:

#         static YCSBWorkload currentWorkload = WORKLOAD_R50;

#     to the requested workload.
#     """
#     if not YCSB_HEADER_LOCAL.exists():
#         raise FileNotFoundError(
#             f"Could not find {YCSB_HEADER_LOCAL}. "
#             "Run this script from the stellar-core repo root."
#         )

#     text = YCSB_HEADER_LOCAL.read_text()

#     new_text, count = re.subn(
#         r"static\s+YCSBWorkload\s+currentWorkload\s*=\s*WORKLOAD_[A-Za-z0-9_]+\s*;",
#         f"static YCSBWorkload currentWorkload = {workload_name};",
#         text,
#         count=1,
#     )

#     if count != 1:
#         raise RuntimeError(
#             "Could not find exactly one currentWorkload line in "
#             f"{YCSB_HEADER_LOCAL}"
#         )

#     YCSB_HEADER_LOCAL.write_text(new_text)
#     print(f"✅ Updated {YCSB_HEADER_LOCAL}: currentWorkload = {workload_name}")


# def fetch_existing_instances():
#     fetch_cmd = f"""
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={project} \
#         --quiet
#     """

#     print(f"🗑️ Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def parse_tsm_index(name):
#     return int(name.rsplit("-", 1)[1])


# def run_command(command):
#     print(f"Running: {command}")
#     return subprocess.call(command, shell=True)


# def kill_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas/stellar-private 2>/dev/null || true; \
# sudo pkill -9 stellar-core || true; sudo pkill -9 shab_client || true; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def clean_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def copy_folder_to_instance(
#     i,
#     source_folder="/home/tejas/stellar-private",
#     destination_path="/home/tejas/stellar-private",
# ):
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}" """

#     print(f"Executing command for {instance_name}: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Command for {instance_name} finished with exit code: {output}")

#     return (instance_name, output)


# def run_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     node_number = i + 1
#     instance_name = f"tsm-sc-{i:03}"

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Return code for {instance_name}: {output}")
#     return output


# def git_pull_stellar(i):
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def compile_stellar_and_client(i):
#     """
#     First subrun only.
#     This compiles both shab_client and stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def copy_ycsb_header_to_instance(i):
#     """
#     Later subruns only.
#     Copy changed CustomYCSBWorkload.h to the VM source tree.
#     This keeps the source file in sync. We only recompile the client binary after this.
#     """
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     remote_path = (
#         f"{instance_name}:/home/tejas/stellar-core/src/overlay/"
#         f"{YCSB_HEADER_LOCAL.name}"
#     )

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{YCSB_HEADER_LOCAL}" "{remote_path}" """

#     print(f"Copying workload header to {instance_name}: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Header copy to {instance_name} finished with exit code: {output}")
#     return (instance_name, output)


# def compile_client_only(i):
#     """
#     Every subrun after the first.
#     This recompiles only shab_client, not stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd /home/tejas/stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# # =============================================================================
# # VM setup
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()

#     print("\n➡ Existing instances to delete before first run:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     if instances:
#         run_parallel(
#             delete_instance,
#             instances,
#             max_workers=min(32, len(instances)),
#         )
#         print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#     else:
#         print("\n✔ No existing tsm-sc-* instances found before first run.\n")


# # Create required replica/client machines.
# existing_instances = fetch_existing_instances()
# existing_names = {name for name, _ in existing_instances}

# commands = []

# # Replicas: tsm-sc-000 ... tsm-sc-007
# for i in range(NUM_NODES):
#     instance_name = f"tsm-sc-{i:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing replica {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# # Client: tsm-sc-008
# for i in range(n_clients):
#     client_idx = NUM_NODES + i
#     instance_name = f"tsm-sc-{client_idx:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing client {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(client_idx)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# if commands:
#     run_parallel(
#         run_command,
#         commands,
#         max_workers=min(48, len(commands)),
#     )

#     print("All missing instances launched.")

#     # Give GCP/SSH a little time after VM creation.
#     time.sleep(30)
# else:
#     print("✔ All required instances already exist; no VM creation needed.")


# # Get sorted node and client IPs.
# ip_cmd = f"""
# gcloud compute instances list \
#     --project={project} \
#     --filter="name~'^tsm-sc-'" \
#     --sort-by=name \
#     --format="value(name,zone,networkInterfaces[0].networkIP)"
# """

# ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

# instance_records = []

# for line in ip_output.splitlines():
#     if line.strip():
#         name, inst_zone, ip = line.split()
#         idx = int(name.rsplit("-", 1)[1])
#         instance_records.append((idx, name, inst_zone, ip))

# instance_records.sort()

# node_records = [r for r in instance_records if r[0] < NUM_NODES]
# client_records = [r for r in instance_records if NUM_NODES <= r[0] < NUM_NODES + n_clients]

# if len(node_records) != NUM_NODES:
#     raise RuntimeError(
#         f"Expected {NUM_NODES} replica nodes, but found {len(node_records)}: {node_records}"
#     )

# if len(client_records) != n_clients:
#     raise RuntimeError(
#         f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#     )

# iplist = [r[3] for r in node_records]

# with open("tsm_ips.txt", "w") as f:
#     for ip in iplist:
#         f.write(ip + "\n")

# print("🎯 Node IPs:", iplist)
# print("🎯 Client instances:", client_records)

# node1_ip = iplist[0]
# print(f"Clients will connect to leader/node1 at: {node1_ip}")

# all_instance_indices = list(range(NUM_NODES + n_clients))


# # =============================================================================
# # Read-ratio experiment loop
# # =============================================================================

# for workload_idx, workload_cfg in enumerate(READ_RATIO_WORKLOADS):
#     read_ratio = workload_cfg["read_ratio"]
#     workload_name = workload_cfg["workload"]

#     print("\n" + "#" * 100)
#     print(
#         f"Starting read-ratio experiment: "
#         f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, workload={workload_name}"
#     )
#     print("#" * 100)

#     # -------------------------------------------------------------------------
#     # Change CustomYCSBWorkload.h for this subrun.
#     # -------------------------------------------------------------------------
#     set_local_ycsb_workload(workload_name)

#     # -------------------------------------------------------------------------
#     # First subrun:
#     #   - commit/push changed workload
#     #   - git pull on every VM
#     #   - compile shab_client + stellar-core on every VM
#     #
#     # Later subruns:
#     #   - copy changed header to VMs
#     #   - compile only shab_client on client VM(s)
#     # -------------------------------------------------------------------------
#     first_subrun = (workload_idx == 0)

#     if first_subrun:
#         subprocess.call(
#             f'git add .; git commit -m "read ratio workload {workload_name}" || true; git push',
#             shell=True,
#         )

#         # Optional local build, preserved from your earlier script.
#         subprocess.call("make -j8", shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             git_pull_stellar,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#         results = run_parallel(
#             compile_stellar_and_client,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)
#     else:
#         # No git pull and no stellar-core compilation after the first subrun.
#         # The server binary is unchanged. The client workload changes through
#         # CustomYCSBWorkload.h, so update source and recompile shab_client.
#         results = run_parallel(
#             copy_ycsb_header_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path("../stellar-private")
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     subprocess.call(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True,
#     )

#     # -------------------------------------------------------------------------
#     # Fixed client load loop.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = (
#             active_clients * client_threads * client_max_in_flight
#         )

#         active_client_indices = [
#             NUM_NODES + j for j in range(active_clients)
#         ]

#         # Compile the client every subrun. On the first subrun it was already
#         # compiled by compile_stellar_and_client(), but compiling it again is
#         # cheap and guarantees the workload header is reflected in shab_client.
#         results = run_parallel(
#             compile_client_only,
#             active_client_indices,
#             max_workers=active_clients,
#         )
#         print(results)

#         run_label = (
#             f"read_ratio_{read_ratio}_"
#             f"{workload_name}_"
#             f"nodes_{NUM_NODES}_"
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(
#             f"🚀 Starting read-ratio run: "
#             f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, "
#             f"workload={workload_name}, {run_label}"
#         )
#         print("=" * 80)

#         results = run_parallel(
#             clean_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             copy_folder_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(NUM_NODES),
#             max_workers=min(48, NUM_NODES),
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial write batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SHABDIZ_vs_read_ratio_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={NUM_NODES}\n")
#             f.write(f"read_ratio_percent={read_ratio}\n")
#             f.write(f"workload={workload_name}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"client_wait_after_start_sec={CLIENT_WAIT_AFTER_START_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")
#             f.write(f"compile_stellar_core_this_subrun={first_subrun}\n")
#             f.write("client_compiled_this_subrun=true\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}" """

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, NUM_NODES)) to range(NUM_NODES) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, NUM_NODES)),
#             max_workers=min(48, max(1, min(3, NUM_NODES))),
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log" """

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")


# # =============================================================================
# # Cleanup after all read-ratio subruns
# # =============================================================================

# # if DELETE_ALL_AFTER_FINAL_RUN:
# #     instances = fetch_existing_instances()

# #     print("\n➡ All read-ratio runs complete. Deleting all remaining tsm-sc-* VMs:")
# #     for name, inst_zone in instances:
# #         print(f"  - {name} ({inst_zone})")

# #     if instances:
# #         run_parallel(
# #             delete_instance,
# #             instances,
# #             max_workers=min(32, len(instances)),
# #         )
# #         print(f"\n🧹 Deleted {len(instances)} instance(s).\n")
# #     else:
# #         print("\n✔ No instances to delete.\n")
# # else:
# #     print("\n✔ All read-ratio runs complete. Leaving VMs running.")



➡ Existing instances to delete before first run:

✔ No existing tsm-sc-* instances found before first run.



Running: gcloud compute instances create tsm-sc-000         --project=research-488322         --zone=us-central1-a         --machine-type=e2-standard-2         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=254510644191-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-ec

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-000].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-000  us-central1-a  e2-standard-2               10.128.0.68  34.70.225.218  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-001].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-006].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-008].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



...

Detected 8 nodes based on tsm_ips.txt.
Initializing database for node1...
Initializing database for node2...
Initializing database for node3...
Initializing database for node4...
Initializing database for node5...
Initializing database for node6...


2026-06-24T13:08:19.870 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2026-06-24T13:08:19.872 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "node7",
      "node6",
      "node2",
      "node5",
      "node8",
      "GDB7U",
      "node3"
   ]
}

2026-06-24T13:08:19.872 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-24T13:08:19.872 [default INFO] Assigning calculated value of 3 to FAILURE_SAFETY
2026-06-24T13:08:19.919 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2026-06-24T13:08:19.921 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "node7",
      "node6",
      "GC4TM",
      "node5",
      "node8",
      "node1",
      "node3"
   ]
}

2026-06-24T13:08:19.921 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
202

Initializing database for node7...
Initializing database for node8...
✅ 8-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node5/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node6/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node7/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node8/ste

2026-06-24T13:08:20.097 [default INFO] Config from /home/tejas/stellar-private/node7/stellar-core.cfg
2026-06-24T13:08:20.099 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "GCDAF",
      "node6",
      "node2",
      "node5",
      "node8",
      "node1",
      "node3"
   ]
}

2026-06-24T13:08:20.099 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-24T13:08:20.099 [default INFO] Assigning calculated value of 3 to FAILURE_SAFETY
2026-06-24T13:08:20.130 [default INFO] Config from /home/tejas/stellar-private/node8/stellar-core.cfg
2026-06-24T13:08:20.133 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "node7",
      "node6",
      "node2",
      "node5",
      "GC73B",
      "node1",
      "node3"
   ]
}

2026-06-24T13:08:20.133 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
202

0
[0]

🚀 Starting read-ratio run: num_nodes=8, read_ratio=40%, workload=WORKLOAD_R40, read_ratio_40_WORKLOAD_R40_nodes_8_clients_1_threads_2_inflight_150_total_threads_2_total_inflight_300
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-000" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-001" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-002" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-003" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-004" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a

#  Memory Exp

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

# #     if do_compile_this_run:
# #         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

# #         n_collection = 100
# #         subprocess.call('make -j8', shell=True)

# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def git_pull_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # git pull"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             git_pull_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )
# #         print(results)

# #         # ---------------------------------------------------------------------
# #         # Compile only on the first/largest run.
# #         # Since later runs reuse a subset of these VMs, no recompilation is needed.
# #         # ---------------------------------------------------------------------
# #         def compile_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # g++ -O2 -std=c++17 -pthread \
# # -I/home/tejas/stellar-core/src \
# # /home/tejas/stellar-core/shab_client.cpp \
# # -o /home/tejas/stellar-core/shab_client; \
# # make -j16; \
# # cd; \
# # sudo rm -rf stellar-private"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             compile_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )


        
# #         print(results)
# #     else:
# #         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

# #     # -------------------------------------------------------------------------
# #     # Generate stellar-private configs locally using only replica IPs.
# #     # -------------------------------------------------------------------------
# #     stellar_private_path = Path('../stellar-private')
# #     if stellar_private_path.exists():
# #         shutil.rmtree(stellar_private_path)
# #     stellar_private_path.mkdir()

# #     subprocess.call(
# #         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     subprocess.call(
# #         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
# #         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     # Enable custom message only on leader.
# #     line_to_add = "SEND_CUSTOM_MESSAGE=true"
# #     target_file = "../stellar-private/node1/stellar-core.cfg"

# #     subprocess.call(
# #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #         shell=True
# #     )

# #     # Optional memory profiling on node2.
# #     if num_nodes >= 2:
# #         target_file = "../stellar-private/node2/stellar-core.cfg"
# #         line_to_add = "MEMORY_PROF=true"

# #         subprocess.call(
# #             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #             shell=True
# #         )

# #     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

# #     # -------------------------------------------------------------------------
# #     # Throughput/latency experiment loop.
# #     # Fixed offered load for scalability:
# #     # active_clients=1, client_threads=2, max_in_flight=100.
# #     # Aggregate max in-flight = 200.
# #     # -------------------------------------------------------------------------
# #     for load in LOAD_POINTS:
# #         active_clients = load["active_clients"]
# #         client_threads = load["client_threads"]
# #         client_max_in_flight = load["max_in_flight"]

# #         if active_clients > n_clients:
# #             raise RuntimeError(
# #                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
# #             )

# #         total_client_threads = active_clients * client_threads
# #         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

# #         run_label = (
# #             f"clients_{active_clients}_threads_{client_threads}_"
# #             f"inflight_{client_max_in_flight}_"
# #             f"total_threads_{total_client_threads}_"
# #             f"total_inflight_{aggregate_max_in_flight}"
# #         )

# #         print("\n" + "=" * 80)
# #         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
# #         print("=" * 80)

# #         def clean_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             remote_command = f"""\
# # cd /home/tejas; \
# # sudo rm -rf stellar-private; \
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")
# #             output = subprocess.call(command, shell=True)
# #             print(f"Return code for tsm-sc-{i:03}: {output}")
# #             return output

# #         results = run_parallel(
# #             clean_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def copy_folder_to_instance(
# #             i,
# #             source_folder="/home/tejas/stellar-private",
# #             destination_path="/home/tejas/stellar-private"
# #         ):
# #             inst_zone = get_zone_for_instance(i)
# #             instance_name = f"tsm-sc-{i:03}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

# #             print(f"Executing command for {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Command for {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         results = run_parallel(
# #             copy_folder_to_instance,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def run_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             node_number = i + 1
# #             instance_name = f"tsm-sc-{i:03}"

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# # > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Kill old processes on all replica and client machines.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         # Start consensus replicas.
# #         results = run_parallel(
# #             run_stellar_private,
# #             range(num_nodes),
# #             max_workers=min(48, num_nodes)
# #         )

# #         print(results)
# #         print("All Stellar nodes should be starting in the background.")

# #         # Give nodes time to authenticate and start the client listener.
# #         time.sleep(80)

# #         def run_stellar_client(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# # {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# # {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# # > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing client command: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Start only the required number of client VMs for this load point.
# #         active_client_indices = [
# #             num_nodes + j for j in range(active_clients)
# #         ]

# #         results = run_parallel(
# #             run_stellar_client,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print(results)
# #         print(
# #             f"Started {active_clients} client VM(s), "
# #             f"each with {client_threads} client threads. "
# #             f"Total client threads = {total_client_threads}. "
# #             f"Aggregate max in-flight = {aggregate_max_in_flight}."
# #         )

# #         # Wait for the duration run to produce stable per-second client logs.
# #         # The clients may wait forever on final partial batches, so we kill them after this.
# #         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

# #         # Stop all nodes and clients.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         remote_base_folder = "/home/tejas/stellar-private"

# #         local_base_destination = (
# #             "/home/tejas/work/experiments/shabdiz/"
# #             + f"memory_no_cleanup_{num_nodes}_{run_label}"
# #         )

# #         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

# #         # Save run metadata.
# #         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
# #             f.write(f"num_nodes={num_nodes}\n")
# #             f.write(f"active_clients={active_clients}\n")
# #             f.write(f"client_threads_per_vm={client_threads}\n")
# #             f.write(f"total_client_threads={total_client_threads}\n")
# #             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
# #             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
# #             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
# #             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
# #             f.write(f"leader_ip={node1_ip}\n")
# #             f.write(f"machine_type={machine_type}\n")
# #             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

# #         def copy_folder_from_instance(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"

# #             node_number = i + 1
# #             node_folder = f"node{node_number}"

# #             remote_source_path = posixpath.join(remote_base_folder, node_folder)

# #             local_destination_path = Path(local_base_destination) / instance_name
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             remote_source = f"{instance_name}:{remote_source_path}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{remote_source}" "{local_destination_path}"'''

# #             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy from {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         # Copy only a few node logs to reduce time.
# #         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
# #         node_copy_results = run_parallel(
# #             copy_folder_from_instance,
# #             range((num_nodes)),
# #             max_workers=min(48, max(1, min(3, num_nodes)))
# #         )

# #         def copy_client_log(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_source = (
# #                 f"{instance_name}:/home/tejas/stellar-private/"
# #                 f"stellar-client-{client_id}.log"
# #             )

# #             local_destination_path = Path(local_base_destination)
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

# #             print(f"Copying client log from {instance_name}...")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy finished with exit code: {output}")

# #             return (instance_name, output)

# #         client_copy_results = run_parallel(
# #             copy_client_log,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print("\n--- Summary of Download Results ---")
# #         print("Node log copies:", node_copy_results)
# #         print("Client log copies:", client_copy_results)
# #         print(f"Saved run to: {local_base_destination}")



#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     if num_nodes >= 2:
#         target_file = "../stellar-private/node2/stellar-core.cfg"
#         line_to_add = "MEMORY_PROF=true"

#         subprocess.call(
#             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#             shell=True
#         )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"memory_withv3_cleanup_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# PBFT Test

In [2]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output




#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )


#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output



#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(40)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"PBFT_test_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")


####################################################################################################
Starting experiment for num_nodes=4
####################################################################################################


Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-a             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm             

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-004].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-000].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-004  us-central1-a  e2-standard-2               10.128.0.10  35.253.53.162  RUNNING


 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



...

/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket/HotArchiveBucketList.o bucket/InMemoryIndex.o bucket/LiveBucket.o bucket/LiveBucketIndex.o bucket/LiveBucketList.o bucket/MergeKey.o bucket/SearchableBucketList.o catchup/ApplyBucketsWork.o catchup/ApplyBufferedLedgersWork.o catchup/ApplyCheckpointWork.o catchup/ApplyLedgerWork.o

2026-06-27T21:32:40.203 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2026-06-27T21:32:40.205 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node3", "GDEBP", "node4", "node2" ]
}

2026-06-27T21:32:40.205 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-27T21:32:40.205 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2026-06-27T21:32:40.257 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2026-06-27T21:32:40.259 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node3", "node1", "node4", "GDKDT" ]
}

2026-06-27T21:32:40.259 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-27T21:32:40.260 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2026-06-27T21:32:40.292 [default INFO] Config from /home/tejas/stellar-private/node3/ste

Initializing database for node3...
Initializing database for node4...
✅ 4-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &

🚀 Starting throughput/latency run: num_nodes=4, clients_1_threads_2_inflight_150_total_threads_2_total_inflight_300
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-000" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-001" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stella

2026-06-27T21:32:40.327 [default INFO] Config from /home/tejas/stellar-private/node4/stellar-core.cfg
2026-06-27T21:32:40.330 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node3", "node1", "GDJTT", "node2" ]
}

2026-06-27T21:32:40.330 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-27T21:32:40.330 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY


Return code for tsm-sc-003: 0
Return code for tsm-sc-000: 0
Return code for tsm-sc-001: 0
Return code for tsm-sc-002: 0
Return code for tsm-sc-004: 0
Executing command for tsm-sc-000: gcloud compute scp --zone "us-central1-a" --project "research-488322" --recurse "/home/tejas/stellar-private" "tsm-sc-000:/home/tejas/stellar-private"
Executing command for tsm-sc-001: gcloud compute scp --zone "us-central1-a" --project "research-488322" --recurse "/home/tejas/stellar-private" "tsm-sc-001:/home/tejas/stellar-private"
Executing command for tsm-sc-002: gcloud compute scp --zone "us-central1-a" --project "research-488322" --recurse "/home/tejas/stellar-private" "tsm-sc-002:/home/tejas/stellar-private"
Executing command for tsm-sc-003: gcloud compute scp --zone "us-central1-a" --project "research-488322" --recurse "/home/tejas/stellar-private" "tsm-sc-003:/home/tejas/stellar-private"
Executing command for tsm-sc-004: gcloud compute scp --zone "us-central1-a" --project "research-488322" --recu

In [8]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


➡ Existing instances to delete:

✔ No tsm-sc-* instances found.

